# Stage 2 Notebook 19 - Exp2N CLRKD with proper top-K decode + lane-NMS lane-line F1

**This is not a new training recipe.** The model is identical to Exp2M (sqrt rescaling on the IoU regression target -- our geometry champion at matched_iou=0.42, point_mae=0.32, with stable cls supervision). What's new is the **evaluation pipeline**.

Across Exp2G/H/I/J/K/L/M (seven cls ablations), `val/lane/clrkd_style_f1` reported 0 in every single experiment. While auditing the metric I found the cause: `CLRKDStyleLaneMetric` uses a hardcoded `score_threshold = 0.5` (`metrics/original_metric_adapters.py:138`). Our models' max sigmoid score has been ~0.15 -- well below 0.5 -- so its `pred_exist` is all-False by construction. The metric ALSO reads `matched_target` (post-matching alignment) instead of original GT, so even with a fixed threshold it's measuring per-slot quality, not lane-line F1.

**Our headline lane-F1 number has been an evaluator bug, not a training failure.** This notebook fixes that by introducing proper inference:

- `stage2/fusion/lane_decode.py`: per-image top-K by descending sigmoid score (no threshold floor by default), then lane-NMS by LineIoU.
- `stage2/metrics/lane_f1_decoded.py`: greedy-by-descending-IoU assignment of decoded lanes to ORIGINAL GT lanes (not matched_target). Returns `lane/decoded_f1`, `lane/decoded_precision`, `lane/decoded_recall`, `lane/decoded_pred_count`, `lane/decoded_avg_score`.

This is the metric that compares directly to CLRKDNet's published lane-line F1 (after dataset normalization). If Exp2N's `decoded_f1` is meaningfully > 0, our model has been quietly working all along and the previous experiments' apparent failure was the metric, not the model. If it's still ~ 0, we know cls scoring genuinely cannot rank priors and we need the next-step fix (extended training or hybrid binary+IoU head).

Reference: external_repos/CLRKDNet-master/clrkd/models/heads/clr_head.py (top-K + NMS), external_repos/CLRNet/clrnet/utils/lane.py (LineIoU NMS).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through the same Exp2M loss path. The new
# decoded metric does not run during smoke (smoke uses random targets and
# the metric is val-only), but the new imports must not break the train
# script's import chain.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp14_rmt_gca_clrkd_decoded_eval_joint_smoke.log
OK exp14_rmt_gca_clrkd_decoded_eval_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=10.3074 det_loss=3.1365 grad_cos=0.0593 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4996342658996582, 'gate/lane_mean': 0.5027599930763245, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [4]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp14_rmt_gca_clrkd_decoded_eval_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp14_rmt_gca_clrkd_decoded_eval_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/data

0

## What to watch in Exp2N training

The `epoch_summary` line now includes new fields:
- `val_decoded_f1` -- the metric directly comparable to CLRKDNet on lane-line F1.
- `val_decoded_p`, `val_decoded_r` -- precision and recall components.
- `decoded_pred` -- average number of decoded lanes per batch (capped at top_k * batch_size).

Pass criteria at epoch 10:

- `val_decoded_pred > 0` for every epoch (sanity).
- `val_decoded_f1 >= 0.10` -- our first non-zero lane-F1 number ever measured.
- Both `val_decoded_p` and `val_decoded_r` non-zero (one-sided result means top_k or threshold mis-configured).
- Geometry holds (Exp2M baseline): `val_lane_point_mae <= 0.34`, `val_matched_line_iou >= 0.40`.

Failure signals -> next ablation:

- `decoded_f1 < 0.05`: cls scoring genuinely cannot rank priors. Move to (a) extended training (30+ epochs since CLRKDNet trains for 100s) or (b) hybrid binary+IoU head with separate losses.
- `decoded_pred == 0` consistently: bug in lane_decode/NMS. Re-derive against CLRKDNet's reference code path.
- Geometry regresses: instrumentation introduced an OOM or batch-skipping side effect. Check the val loop.

After short10, run NB08 to plot Exp2K / Exp2L / Exp2M / Exp2N side-by-side, with the new `lane/decoded_f1` curve included.